In [3]:
from IPython.core.display import display, HTML
display(HTML("<style>.jp-Cell { width: 140% !important; }</style>"))


/tmp/ipykernel_30315/116758188.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from gensim.models import Word2Vec
from scipy.linalg import orthogonal_procrustes
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings('ignore')

# # =============================================================================
# # CONFIGURATION
# # =============================================================================

# Measure drift for target words
target_words = ['neural', 'agent', 'memory', 'cell', 'network']
target_words_guardian = ['cloud', 'stream']


# SEEDS        = [42, 123, 777]
# # SEEDS        = [777]
# N_ANCHORS    = 3000   # How many anchor words to use for alignment.
#                        # More = better alignment but slower.

# # File paths — adjust if your .bin files live elsewhere
# ARXIV_FILES = {
#     '1990-2011': {42: './CS_arXiv/u_model_1990_42.bin',  123: './CS_arXiv/u_model_1990_123.bin',  777: './CS_arXiv/u_model_1990_777.bin'},
#     '2012-2019': {42: './CS_arXiv/u_model_2012_42.bin',  123: './CS_arXiv/u_model_2012_123.bin',  777: './CS_arXiv/u_model_2012_777.bin'},
#     '2020-2023': {42: './CS_arXiv/u_model_2020_42.bin',  123: './CS_arXiv/u_model_2020_123.bin',  777: './CS_arXiv/u_model_2020_777.bin'},
#     # '2000-2009': {777: 'model_2000_777.bin'},
#     # '2012-2019': {777: 'model_2012_777.bin'},
#     # '2020-2025': {777: 'model_2020_777.bin'},

# }
# PUBMED_FILES = {
#     42:  './pubmed/model_pubmed_200_42.bin',
#     123: './pubmed/model_pubmed_200_123.bin',
#     777: './pubmed/model_pubmed_200_777.bin',
# }

# GUARDIAN_FILES = {
#     42:  './guardian/guardian_42.bin',
#     123: './guardian/guardian_123.bin',
#     777: './guardian/guardian_777.bin',
# }

# # NEWS_FILES = {
# #     42:  'news_corpus_description_42.bin',
# #     123: 'news_corpus_description_123.bin',
# #     777: 'news_corpus_description_777.bin',
# # }

# PERIOD_LABELS = list(ARXIV_FILES.keys())
# TRANSITIONS   = [
#     ('1990-2011', '2012-2019', '1990 → 2010s'),
#     ('2012-2019', '2020-2023', '2010s → 2020s'),
# ]

In [5]:
def get_anchors(model_a, model_b, exclude_words=None, min_count=500):
    """
    Compute the shared anchor word list between two models.
    Separated from alignment so the SAME anchor set can be reused
    across multiple alignment calls — critical for fair SNR comparison.
    """
    exclude = set(exclude_words or [])
    vocab_a  = set(model_a.wv.index_to_key)
    vocab_b  = set(model_b.wv.index_to_key)
    shared   = sorted(vocab_a & vocab_b - exclude)
    shared   = [w for w in shared
                if model_a.wv.get_vecattr(w, "count") > min_count
                and model_b.wv.get_vecattr(w, "count") > min_count]
    return shared


def fit_procrustes(model_a, model_b, anchors):
    """
    Fit rotation R using the given anchor list.
    Returns R, src_mean — the two things needed to transform
    any source vector into the target space.
    """
    A = np.array([model_a.wv[w] for w in anchors])
    B = np.array([model_b.wv[w] for w in anchors])

    A /= np.linalg.norm(A, axis=1, keepdims=True)
    B /= np.linalg.norm(B, axis=1, keepdims=True)

    src_mean = A.mean(axis=0)
    tgt_mean = B.mean(axis=0)

    A -= src_mean
    B -= tgt_mean

    R, _ = orthogonal_procrustes(A, B)

    cos_errors = [
        1 - np.dot((A[i] @ R), B[i]) /
        (np.linalg.norm(A[i] @ R) * np.linalg.norm(B[i]))
        for i in range(len(anchors))
    ]
    residual = np.mean(cos_errors)

    return R, src_mean, tgt_mean, residual


def apply_procrustes(source_model, R, src_mean, tgt_mean):
    """
    Apply a pre-fitted rotation to ALL vectors in source_model.
    Applies the same normalise → centre → rotate pipeline
    that was used during fitting.
    """
    all_words   = source_model.wv.index_to_key
    all_vectors = np.array([source_model.wv[w] for w in all_words])

    norms       = np.linalg.norm(all_vectors, axis=1, keepdims=True)
    norms       = np.where(norms == 0, 1, norms)
    all_vectors = all_vectors / norms

    all_vectors -= src_mean

    aligned = all_vectors @ R + tgt_mean
    return aligned

In [6]:
# ── 1. Load models ─────────────────────────────────────────────────────────────
arxiv_2020_42 = Word2Vec.load("./CS_arXiv/u_model_2020_42.bin")
arxiv_2012_42 = Word2Vec.load("./CS_arXiv/u_model_2012_42.bin")
arxiv_1990_42 = Word2Vec.load("./CS_arXiv/u_model_1990_42.bin")
pubmed_42     = Word2Vec.load("./pubmed/model_pubmed_200_42.bin")
guardian_42   = Word2Vec.load("./guardian/guardian_42.bin")
print('loading models....')

# ── 1. Load models 123 ─────────────────────────────────────────────────────────────
arxiv_2020_123 = Word2Vec.load("./CS_arXiv/u_model_2020_123.bin")
arxiv_2012_123 = Word2Vec.load("./CS_arXiv/u_model_2012_123.bin")
arxiv_1990_123 = Word2Vec.load("./CS_arXiv/u_model_1990_123.bin")
pubmed_123     = Word2Vec.load("./pubmed/model_pubmed_200_123.bin")
guardian_123   = Word2Vec.load("./guardian/guardian_123.bin")
print('loading models....')

# ── 1. Load models 777 ─────────────────────────────────────────────────────────────
arxiv_2020_777 = Word2Vec.load("./CS_arXiv/u_model_2020_777.bin")
arxiv_2012_777 = Word2Vec.load("./CS_arXiv/u_model_2012_777.bin")
arxiv_1990_777 = Word2Vec.load("./CS_arXiv/u_model_1990_777.bin")
pubmed_777    = Word2Vec.load("./pubmed/model_pubmed_200_777.bin")
guardian_777 = Word2Vec.load("./guardian/guardian_777.bin")

print('finished loading models....')

loading models....
loading models....
finished loading models....


In [7]:
exclude = set(target_words + target_words_guardian)

def start_alignment(arxiv_42, arxiv_123, arxiv_777, target_model_42, target_model_123, target_model_777):

    cross_anchors = get_anchors(arxiv_42, target_model_42, exclude)
    noise_anchors = [w for w in cross_anchors
                          if w in arxiv_42.wv
                          and w in arxiv_123.wv
                          and arxiv_42.wv.get_vecattr(w, "count") > 500
                          and arxiv_123.wv.get_vecattr(w, "count") > 500]
    
    _, _, _, eps_42_123 = fit_procrustes(arxiv_42,  arxiv_123, noise_anchors)
    _, _, _, eps_42_777 = fit_procrustes(arxiv_42,  arxiv_777, noise_anchors)
    _, _, _, eps_123_777= fit_procrustes(arxiv_123, arxiv_777, noise_anchors)
    epsilon = np.mean([eps_42_123, eps_42_777, eps_123_777])
    
    R_42,  mean_42, tgt_mean_42,  res_42  = fit_procrustes(arxiv_42,  target_model_42,  cross_anchors)
    R_123, mean_123, tgt_mean_123, res_123 = fit_procrustes(arxiv_123, target_model_123, cross_anchors)
    R_777, mean_777, tgt_mean_777, res_777 = fit_procrustes(arxiv_777, target_model_777, cross_anchors)
    
    cross_residual = np.mean([res_42, res_123, res_777])
    snr = cross_residual / epsilon
    
    print(f"Anchors used       : {len(cross_anchors)}")
    print(f"Noise floor ε      : {epsilon:.4f}")
    print(f"Cross-domain resid : {cross_residual:.4f}")
    print(f"SNR                : {snr:.2f}×")
    
    # ── Apply to get aligned vectors for drift scoring ─────────────────────────────
    aligned_model_42  = apply_procrustes(arxiv_42,  R_42,  mean_42, tgt_mean_42)
    aligned_model_123 = apply_procrustes(arxiv_123, R_123, mean_123, tgt_mean_123)
    aligned_model_777 = apply_procrustes(arxiv_777, R_777, mean_777, tgt_mean_777)

    return aligned_model_42, aligned_model_123, aligned_model_777

In [48]:
def cross_domain_snr_table(words, aligned_42, vocab_42, aligned_123, vocab_123,
                            aligned_777, vocab_777, target_model, epsilon, label):
    print(f"═════════════════════════════════════════════════════════════════")
    print(f" {label}")
    print(f" Noise floor ε      : {epsilon:.4f}")
    print(f"─────────────────────────────────────────────────────────────────")
    print(f"{'Word':<15}{'Mean Dist':>10}{'Std':>10}{'SNR':>10}   Signal?")
    print(f"─────────────────────────────────────────────────────────────────")

    for word in words:
        dists = [
            cross_domain_distance(word, aligned_42,  vocab_42,  target_model[0]),
            cross_domain_distance(word, aligned_123, vocab_123, target_model[1]),
            cross_domain_distance(word, aligned_777, vocab_777, target_model[2]),
        ]
        if None in dists:
            print(f"{word:<15}{'ABSENT':>10}")
            continue
        mean_dist = np.mean(dists)
        std_dist  = np.std(dists)
        snr       = mean_dist / epsilon
        flag      = "✓ above ε" if snr >= 1.5 else ("~ marginal" if snr >= 1.0 else "✗ below ε")
        print(f"{word:<15}{mean_dist:>10.4f}{std_dist:>10.4f}{snr:>10.2f}   {flag}")

    print(f"═════════════════════════════════════════════════════════════════\n")

In [33]:
aligned_pubmed_2012_42, aligned_pubmed_2012_123, aligned_pubmed_2012_777 = start_alignment(arxiv_2012_42, arxiv_2012_123, arxiv_2012_777, pubmed_42, pubmed_123, pubmed_777)

Anchors used       : 1661
Noise floor ε      : 0.0987
Cross-domain resid : 0.5140
SNR                : 5.21×


In [34]:
aligned_pubmed_1990_42, aligned_pubmed_1990_123, aligned_pubmed_1990_777 = start_alignment(arxiv_1990_42, arxiv_1990_123, arxiv_1990_777, pubmed_42, pubmed_123, pubmed_777)

Anchors used       : 512
Noise floor ε      : 0.1144
Cross-domain resid : 0.3971
SNR                : 3.47×


In [35]:
aligned_pubmed_2020_42, aligned_pubmed_2020_123, aligned_pubmed_2020_777 = start_alignment(arxiv_2020_42, arxiv_2020_123, arxiv_2020_777, pubmed_42, pubmed_123, pubmed_777)

Anchors used       : 1883
Noise floor ε      : 0.0835
Cross-domain resid : 0.5149
SNR                : 6.16×


In [49]:
cross_domain_snr_table(
    words=["neural", "memory", "cell", "agent", "network"],
    aligned_42=aligned_pubmed_1990_42,   vocab_42=arxiv_1990_42.wv.index_to_key,
    aligned_123=aligned_pubmed_1990_123, vocab_123=arxiv_1990_123.wv.index_to_key,
    aligned_777=aligned_pubmed_1990_777, vocab_777=arxiv_1990_777.wv.index_to_key,
    target_model= [pubmed_42, pubmed_123, pubmed_777], 
    epsilon=0.1144,   # from the 1990 start_alignment printout
    label="arXiv 1990 → PubMed"
)

cross_domain_snr_table(
    words=["neural", "memory", "cell", "agent", "network"],
    aligned_42=aligned_pubmed_2012_42,   vocab_42=arxiv_2012_42.wv.index_to_key,
    aligned_123=aligned_pubmed_2012_123, vocab_123=arxiv_2012_123.wv.index_to_key,
    aligned_777=aligned_pubmed_2012_777, vocab_777=arxiv_2012_777.wv.index_to_key,
    target_model= [pubmed_42, pubmed_123, pubmed_777], 
    epsilon=0.0987,   # from the 1990 start_alignment printout
    label="arXiv 2012 → PubMed"
)


cross_domain_snr_table(
    words=["neural", "memory", "cell", "agent", "network"],
    aligned_42=aligned_pubmed_2020_42,   vocab_42=arxiv_2020_42.wv.index_to_key,
    aligned_123=aligned_pubmed_2020_123, vocab_123=arxiv_2020_123.wv.index_to_key,
    aligned_777=aligned_pubmed_2020_777, vocab_777=arxiv_2020_777.wv.index_to_key,
    target_model= [pubmed_42, pubmed_123, pubmed_777], 
    epsilon=0.0835,   # from the 1990 start_alignment printout
    label="arXiv 2020 → PubMed"
)

═════════════════════════════════════════════════════════════════
 arXiv 1990 → PubMed
 Noise floor ε      : 0.1144
─────────────────────────────────────────────────────────────────
Word            Mean Dist       Std       SNR   Signal?
─────────────────────────────────────────────────────────────────
neural             0.7987    0.0323      6.98   ✓ above ε
memory             0.7920    0.0460      6.92   ✓ above ε
cell               0.7580    0.0474      6.63   ✓ above ε
agent              0.6643    0.0280      5.81   ✓ above ε
network            0.7633    0.0367      6.67   ✓ above ε
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
 arXiv 2012 → PubMed
 Noise floor ε      : 0.0987
─────────────────────────────────────────────────────────────────
Word            Mean Dist       Std       SNR   Signal?
─────────────────────────────────────────────────────────────────
neural             0.6672    0.0041

In [20]:
aligned_guardian_1990_42, aligned_guardian_1990_123, aligned_guardian_1990_777 = start_alignment(arxiv_1990_42, arxiv_1990_123, arxiv_1990_777, guardian_42, guardian_123, guardian_777)
aligned_guardian_2012_42, aligned_guardian_2012_123, aligned_guardian_2012_777 =start_alignment(arxiv_2012_42, arxiv_2012_123, arxiv_2012_777, guardian_42, guardian_123, guardian_777)
aligned_guardian_2020_42, aligned_guardian_2020_123, aligned_guardian_2020_777 = start_alignment(arxiv_2020_42, arxiv_2020_123, arxiv_2020_777, guardian_42, guardian_123, guardian_777)

Anchors used       : 556
Noise floor ε      : 0.1165
Cross-domain resid : 0.4156
SNR                : 3.57×
Anchors used       : 2120
Noise floor ε      : 0.1021
Cross-domain resid : 0.5229
SNR                : 5.12×
Anchors used       : 2542
Noise floor ε      : 0.0878
Cross-domain resid : 0.5189
SNR                : 5.91×


In [51]:
cross_domain_snr_table(
    words=["cloud", "stream", "training"],
    aligned_42=aligned_guardian_1990_42,   vocab_42=arxiv_1990_42.wv.index_to_key,
    aligned_123=aligned_guardian_1990_123, vocab_123=arxiv_1990_123.wv.index_to_key,
    aligned_777=aligned_guardian_1990_777, vocab_777=arxiv_1990_777.wv.index_to_key,
    target_model= [guardian_42, guardian_123, guardian_777], 
    epsilon=0.1165,   # from the 1990 start_alignment printout
    label="arXiv 1990 → guardian"
)
cross_domain_snr_table(
    words=["cloud", "stream", "training"],
    aligned_42=aligned_guardian_2012_42,   vocab_42=arxiv_2012_42.wv.index_to_key,
    aligned_123=aligned_guardian_2012_123, vocab_123=arxiv_2012_123.wv.index_to_key,
    aligned_777=aligned_guardian_2012_777, vocab_777=arxiv_2012_777.wv.index_to_key,
    target_model= [guardian_42, guardian_123, guardian_777], 
    epsilon=0.1021,   # from the 1990 start_alignment printout
    label="arXiv 2012 → guardian"
)
cross_domain_snr_table(
    words=["cloud", "stream", "training"],
    aligned_42=aligned_guardian_2020_42,   vocab_42=arxiv_2020_42.wv.index_to_key,
    aligned_123=aligned_guardian_2020_123, vocab_123=arxiv_2020_123.wv.index_to_key,
    aligned_777=aligned_guardian_2020_777, vocab_777=arxiv_2020_777.wv.index_to_key,
    target_model= [guardian_42, guardian_123, guardian_777], 
    epsilon=0.0878,   # from the 1990 start_alignment printout
    label="arXiv 2020 → guardian"
)

═════════════════════════════════════════════════════════════════
 arXiv 1990 → guardian
 Noise floor ε      : 0.1165
─────────────────────────────────────────────────────────────────
Word            Mean Dist       Std       SNR   Signal?
─────────────────────────────────────────────────────────────────
cloud              0.7632    0.0360      6.55   ✓ above ε
stream             0.8014    0.0329      6.88   ✓ above ε
training           0.7219    0.0424      6.20   ✓ above ε
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
 arXiv 2012 → guardian
 Noise floor ε      : 0.1021
─────────────────────────────────────────────────────────────────
Word            Mean Dist       Std       SNR   Signal?
─────────────────────────────────────────────────────────────────
cloud              0.5994    0.0091      5.87   ✓ above ε
stream             0.5875    0.0272      5.75   ✓ above ε
training           0.4831    0.

In [31]:
def format_score(score):
    return f"{score:.4f}" if score else "N/A"
    
def cross_domain_distance(word, aligned_matrix, aligned_vocab, target_model):
    """Cosine distance between aligned source vector and target vector."""
    if word not in aligned_vocab or word not in target_model.wv:
        return None
    idx        = aligned_vocab.index(word)
    src_vec    = aligned_matrix[idx]
    tgt_vec    = target_model.wv[word]
    return cosine(src_vec, tgt_vec)

In [32]:
# Measure drift for target words
target_words = ['neural', 'memory', 'cell', 'agent','network']
target_words_guardian = ['cloud','stream','training']

print("\n── Cross-domain drift (arXiv 1990 → PubMed) ──────────────")
print(f"{'Word':<15} {'seed':>10} {'→ 1990':>15} {'→ 2012':>15} {'→ 2020':>15}")
for w in target_words:
    print('--'*39)

    d_pm_1990_42 = cross_domain_distance(w, aligned_pubmed_1990_42, list(arxiv_1990_42.wv.index_to_key), pubmed_42)
    d_pm_1990_123 = cross_domain_distance(w, aligned_pubmed_1990_123, list(arxiv_1990_123.wv.index_to_key), pubmed_123)
    d_pm_1990_777 = cross_domain_distance(w, aligned_pubmed_1990_777, list(arxiv_1990_777.wv.index_to_key), pubmed_777)

    d_pm_2012_42 = cross_domain_distance(w, aligned_pubmed_2012_42, list(arxiv_2012_42.wv.index_to_key), pubmed_42)
    d_pm_2012_123 = cross_domain_distance(w, aligned_pubmed_2012_123, list(arxiv_2012_123.wv.index_to_key), pubmed_123)
    d_pm_2012_777 = cross_domain_distance(w, aligned_pubmed_2012_777, list(arxiv_2012_777.wv.index_to_key), pubmed_777)

    d_pm_2020_42 = cross_domain_distance(w, aligned_pubmed_2020_42, list(arxiv_2020_42.wv.index_to_key), pubmed_42)
    d_pm_2020_123 = cross_domain_distance(w, aligned_pubmed_2020_123, list(arxiv_2020_123.wv.index_to_key), pubmed_123)
    d_pm_2020_777 = cross_domain_distance(w, aligned_pubmed_2020_777, list(arxiv_2020_777.wv.index_to_key), pubmed_777)

    print(f"{w:<15} {42:>9} {format_score(d_pm_1990_42):>16} {format_score(d_pm_2012_42):>15} {format_score(d_pm_2020_42):>15}")
    print(f"{w:<15} {123:>10} {format_score(d_pm_1990_123):>15} {format_score(d_pm_2012_123):>15} {format_score(d_pm_2020_123):>15}")
    print(f"{w:<15} {777:>10} {format_score(d_pm_1990_777):>15} {format_score(d_pm_2012_777):>15} {format_score(d_pm_2020_777):>15}")
    

print("\n── Cross-domain drift (arXiv 1990 → Guardian) ──────────────")
print(f"{'Word':<15} {'seed':>10} {'→ 1990':>15} {'→ 2012':>15} {'→ 2020':>15}")
for w in target_words_guardian:
    print('--'*39)

    d_gd_1990_42 = cross_domain_distance(w, aligned_guardian_1990_42, list(arxiv_1990_42.wv.index_to_key), guardian_42)
    d_gd_1990_123 = cross_domain_distance(w, aligned_guardian_1990_123, list(arxiv_1990_123.wv.index_to_key), guardian_123)
    d_gd_1990_777 = cross_domain_distance(w, aligned_guardian_1990_777, list(arxiv_1990_777.wv.index_to_key), guardian_777)

    d_gd_2012_42 = cross_domain_distance(w, aligned_guardian_2012_42, list(arxiv_2012_42.wv.index_to_key), guardian_42)
    d_gd_2012_123 = cross_domain_distance(w, aligned_guardian_2012_123, list(arxiv_2012_123.wv.index_to_key), guardian_123)
    d_gd_2012_777 = cross_domain_distance(w, aligned_guardian_2012_777, list(arxiv_2012_777.wv.index_to_key), guardian_777)

    d_gd_2020_42 = cross_domain_distance(w, aligned_guardian_2020_42, list(arxiv_2020_42.wv.index_to_key), guardian_42)
    d_gd_2020_123 = cross_domain_distance(w, aligned_guardian_2020_123, list(arxiv_2020_123.wv.index_to_key), guardian_123)
    d_gd_2020_777 = cross_domain_distance(w, aligned_guardian_2020_777, list(arxiv_2020_777.wv.index_to_key), guardian_777)

    print(f"{w:<15} {42:>9} {format_score(d_gd_1990_42):>16} {format_score(d_gd_2012_42):>15} {format_score(d_gd_2020_42):>15}")
    print(f"{w:<15} {123:>10} {format_score(d_gd_1990_123):>15} {format_score(d_gd_2012_123):>15} {format_score(d_gd_2020_123):>15}")
    print(f"{w:<15} {777:>10} {format_score(d_gd_1990_777):>15} {format_score(d_gd_2012_777):>15} {format_score(d_gd_2020_777):>15}")




── Cross-domain drift (arXiv 1990 → PubMed) ──────────────
Word                  seed          → 1990          → 2012          → 2020
------------------------------------------------------------------------------
neural                 42           0.8381          0.6615          0.7509
neural                 123          0.7589          0.6693          0.6257
neural                 777          0.7990          0.6708          0.6655
------------------------------------------------------------------------------
memory                 42           0.7312          0.7695          0.7344
memory                 123          0.8423          0.8126          0.7649
memory                 777          0.8024          0.7625          0.7596
------------------------------------------------------------------------------
cell                   42           0.7109          0.5947          0.5096
cell                   123          0.7404          0.6258          0.5521
cell                   777  

In [11]:
import numpy as np
from scipy.spatial.distance import cosine

# ── Target words ───────────────────────────────────────────────────────────────
target_words          = ['reward','attention','agent','cloud','container','pipeline','hallucination','transformer']

# ── Step 1: Compute temporal anchor sets ──────────────────────────────────────
anchors_1990_2012 = get_anchors(arxiv_1990_42, arxiv_2012_42,
                                exclude_words=target_words, min_count=500)
anchors_2012_2020 = get_anchors(arxiv_2012_42, arxiv_2020_42,
                                exclude_words=target_words, min_count=500)
anchors_1990_2020 = get_anchors(arxiv_1990_42, arxiv_2020_42,
                                exclude_words=target_words, min_count=500)

print(f"1990→2012 anchors : {len(anchors_1990_2012)}")
print(f"2012→2020 anchors : {len(anchors_2012_2020)}")
print(f"1990→2020 anchors : {len(anchors_1990_2020)}")


# ── Step 2: Noise floor for each period transition ────────────────────────────
def compute_epsilon(model_a_42, model_a_123, model_a_777, anchors):
    """
    Noise floor for a single time period using its three seed models.
    Runs all three cross-seed pairs and returns mean residual.
    """
    _, _, _, e1 = fit_procrustes(model_a_42,  model_a_123, anchors)
    _, _, _, e2 = fit_procrustes(model_a_42,  model_a_777, anchors)
    _, _, _, e3 = fit_procrustes(model_a_123, model_a_777, anchors)
    eps = np.mean([e1, e2, e3])
    print(f"  ε (seed noise) : {e1:.4f}  {e2:.4f}  {e3:.4f}  → mean={eps:.4f}")
    return eps

print("\nNoise floor 1990 (anchors from 1990→2012 set):")
eps_1990 = compute_epsilon(arxiv_1990_42, arxiv_1990_123, arxiv_1990_777,
                           anchors_1990_2012)

print("\nNoise floor 2012 (anchors from 1990→2012 set):")
eps_2012 = compute_epsilon(arxiv_2012_42, arxiv_2012_123, arxiv_2012_777,
                           anchors_1990_2012)

print("\nNoise floor 2020 (anchors from 2012→2020 set):")
eps_2020 = compute_epsilon(arxiv_2020_42, arxiv_2020_123, arxiv_2020_777,
                           anchors_2012_2020)


# ── Step 3: Fit temporal alignments for all seed pairs ────────────────────────
def fit_all_seeds(src_models, tgt_models, anchors, label):
    """
    src_models / tgt_models : dicts keyed by seed, e.g. {42: model, 123: ..., 777: ...}
    Returns aligned vectors and residuals for all three seed pairs.
    """
    seeds   = [42, 123, 777]
    results = {}
    resids  = []
    for s in seeds:
        R, mean, tgt_mean, resid = fit_procrustes(src_models[s], tgt_models[s], anchors)
        aligned = apply_procrustes(src_models[s], R, mean, tgt_mean)
        results[s] = {
            'aligned' : aligned,
            'words'   : src_models[s].wv.index_to_key,
            'R'       : R,
            'mean'    : mean,
            'resid'   : resid,
            'tgt_model': tgt_models[s]
        }
        resids.append(resid)
        print(f"  {label} seed={s}  residual={resid:.4f}")
    print(f"  {label} mean residual = {np.mean(resids):.4f}\n")
    return results, np.mean(resids)


arxiv_1990 = {42: arxiv_1990_42, 123: arxiv_1990_123, 777: arxiv_1990_777}
arxiv_2012 = {42: arxiv_2012_42, 123: arxiv_2012_123, 777: arxiv_2012_777}
arxiv_2020 = {42: arxiv_2020_42, 123: arxiv_2020_123, 777: arxiv_2020_777}

print("Temporal alignment 1990 → 2012:")
res_1990_2012, mean_resid_1990_2012 = fit_all_seeds(
    arxiv_1990, arxiv_2012, anchors_1990_2012, "1990→2012")

print("Temporal alignment 2012 → 2020:")
res_2012_2020, mean_resid_2012_2020 = fit_all_seeds(
    arxiv_2012, arxiv_2020, anchors_2012_2020, "2012→2020")

print("Temporal alignment 1990 → 2020 (long-range):")
res_1990_2020, mean_resid_1990_2020 = fit_all_seeds(
    arxiv_1990, arxiv_2020, anchors_1990_2020, "1990→2020")


# ── Step 4: Measure per-word drift ────────────────────────────────────────────

def word_drift_across_seeds(word, alignment_results):
    """
    For a given target word, compute cosine distance between
    its aligned source vector and its raw target vector,
    averaged across the three seed pairs.

    Returns (mean_drift, std_drift, per_seed_distances)
    """
    dists = []
    for s, r in alignment_results.items():
        if word not in r['words']:
            continue
        if word not in r['tgt_model'].wv:
            continue

        idx      = r['words'].index(word)
        src_vec  = r['aligned'][idx]
        tgt_vec  = r['tgt_model'].wv[word]

        # Normalise target vector to match preprocessing applied to source
        tgt_vec  = tgt_vec / np.linalg.norm(tgt_vec)

        dist     = cosine(src_vec, tgt_vec)
        dists.append(dist)

    if not dists:
        return None, None, []
    return np.mean(dists), np.std(dists), dists


# ── Step 5: Report results with SNR ───────────────────────────────────────────

def report_drift(alignment_results, mean_resid, eps_src, eps_tgt,
                 words, transition_label):
    epsilon = max(eps_src, eps_tgt)
    print(f"\n{'═'*65}")
    print(f" {transition_label}")
    print(f" Alignment residual : {mean_resid:.4f}")
    print(f" Noise floor ε      : {epsilon:.4f}")
    print(f" SNR                : {mean_resid/epsilon:.2f}×")
    print(f"{'─'*65}")
    print(f"{'Word':<18} {'Drift':>8} {'Std':>8} {'SNR':>8}  {'Signal?':>10}")
    print(f"{'─'*65}")

    rows = []
    for w in words:
        mean_d, std_d, raw = word_drift_across_seeds(w, alignment_results)
        if mean_d is None:
            print(f"{w:<18} {'ABSENT':>8}")
            continue
        snr   = mean_d / epsilon
        flag  = "✓ above ε" if snr > 1.5 else ("~ marginal" if snr > 1.0 else "✗ noise")
        rows.append((w, mean_d, std_d, snr, flag))
        print(f"{w:<18} {mean_d:>8.4f} {std_d:>8.4f} {snr:>8.2f}  {flag:>10}")

    print(f"{'═'*65}")
    return rows


print("\n\n── TEMPORAL DRIFT RESULTS ────────────────────────────────────────────")

rows_1990_2012 = report_drift(
    res_1990_2012, mean_resid_1990_2012, eps_1990, eps_2012,
    target_words, "1990 → 2012")

rows_2012_2020 = report_drift(
    res_2012_2020, mean_resid_2012_2020, eps_2012, eps_2020,
    target_words, "2012 → 2020")

rows_1990_2020 = report_drift(
    res_1990_2020, mean_resid_1990_2020, eps_1990, eps_2020,
    target_words, "1990 → 2020  [long-range]")

1990→2012 anchors : 653
2012→2020 anchors : 2782
1990→2020 anchors : 652

Noise floor 1990 (anchors from 1990→2012 set):
  ε (seed noise) : 0.1217  0.1224  0.1233  → mean=0.1225

Noise floor 2012 (anchors from 1990→2012 set):
  ε (seed noise) : 0.0673  0.0689  0.0675  → mean=0.0679

Noise floor 2020 (anchors from 2012→2020 set):
  ε (seed noise) : 0.0856  0.0868  0.0868  → mean=0.0864
Temporal alignment 1990 → 2012:
  1990→2012 seed=42  residual=0.2288
  1990→2012 seed=123  residual=0.2284
  1990→2012 seed=777  residual=0.2295
  1990→2012 mean residual = 0.2289

Temporal alignment 2012 → 2020:
  2012→2020 seed=42  residual=0.2772
  2012→2020 seed=123  residual=0.2769
  2012→2020 seed=777  residual=0.2760
  2012→2020 mean residual = 0.2767

Temporal alignment 1990 → 2020 (long-range):
  1990→2020 seed=42  residual=0.2822
  1990→2020 seed=123  residual=0.2827
  1990→2020 seed=777  residual=0.2822
  1990→2020 mean residual = 0.2824



── TEMPORAL DRIFT RESULTS ────────────────────────────